In [1]:
import numpy as np
import matplotlib.pyplot as plt
from Scripts.beams import Beam
from Scripts.beams import BeamType

In [2]:
def beam_interpolate_matrix(x, l):
    xi = x / l
    return np.array([1 - 3 * xi ** 2 + 2 * xi ** 3,  # N1
                     x * (1 - 2 * xi + x ** 2),  # N2
                     3 * xi ** 2 - 2 * xi ** 3,  # N3
                     x * (xi ** 2 - xi)])  # N4


def beam_interpolate_matrix_d1(x, l):
    xi = x / l
    return np.array([(1 / l) * (-6 * xi + 6 * xi ** 2),  # N1
                     (1 - 4 * xi + 3 * xi ** 2),  # N2
                     (1 / l) * (+6 * xi - 6 * xi ** 2),  # N3
                     x * (xi ** 2 - xi)])  # N4    


def beam_interpolate_matrix_d2(x, l):
    xi = x / l
    return np.array([(1 / l) ** 2 * (-6 + 12 * xi),  # N1
                     (1 / l) * (-4 + 6 * xi),  # N2
                     (1 / l) ** 2 * (6 - 12 * xi),  # N3
                     (1 / l) * (-2 + 6 * xi)])  # N4   


def beam_interpolate_matrix_d3(x, l):
    xi = x / l
    return np.array([((1 / l) ** 3) * 12,  # N1
                     ((1 / l) ** 2) * 6,  # N2
                     ((1 / l) ** 3) * -12,  # N3
                     ((1 / l) ** 2) * 6])  # N4   


def beam_euler_stiffness_matrix(E, I, L):
    k = np.array([[12, 6 * L, -12, 6 * L],
                  [6 * L, 4 * L ** 2, -6 * L, 2 * L ** 2],
                  [-12, -6 * L, 12, -6 * L],
                  [6 * L, 2 * L ** 2, -6 * L, 4 * L ** 2]])
    return ((E * I) / L ** 3) * k


def fixed_wall_deform_calc(beam, force, force_dir, force_loc):
    L = beam.length
    E = beam.beam_type.material["elastic modulus"]
    
    Ix = beam.beam_type.cross_section_properties['second moment of area x']
    Iy = beam.beam_type.cross_section_properties['second moment of area y']
    Iz = beam.beam_type.cross_section_properties['second moment of area z']

    K = beam_euler_stiffness_matrix(E, Iz, L)

    f = np.array([-force_dir[2], force_dir[2] * force_loc, 0, 0])
    print(K)
    print(f)
    S = np.linalg.solve(K, f)

    print(S)
    #N1_z = 0
    #N2_z = 0
    #N3_z = 0
    #N4_z = 0
    return S


def plot_deformation(N):
    y = []
    x = range(0, 100)
    for x_val in x:
        y.append(N[0] * x_val ** 3 + N[1] * x_val ** 2 + N[2] * x_val ** 1 + N[3])
    plt.plot(x, y)


beam_type_1 = BeamType("annulus", [0.005, 0.0045], "Aluminum7075-T6", "type_1")
beam_type_2 = BeamType("annulus", [0.003, 0.0025], "Aluminum7075-T6", "type_2")

beam_1 = Beam(beam_type_1, [0, 0, 0], [1, 0, 0], "beam_1")
beam_2 = Beam(beam_type_2, [0, 0, 0], [2, 0, 0], "beam_2")

force = 100
force_dir = [0, 0, -1]

calc_1_z = fixed_wall_deform_calc(beam_1, force, force_dir, force_loc)
calc_2_z = fixed_wall_deform_calc(beam_2, force, force_dir, force_loc)

plot_deformation(calc_1_z)
plot_deformation(calc_2_z)

NameError: name 'force_loc' is not defined